# Модуль 02 · Урок 4. Условные столбцы, where, mask и assign

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Столбец как результат вычисления
Математические действия с Series выполняются **поэлементно**: `df['hours'] * df['rate']` сразу рассчитывает оплату всех работников. Для добавления колонки без изменения исходного DataFrame удобен `assign(new_col=...)`. Если колонка вычисляется из других, можно передать функцию `lambda frame: ...`, особенно в цепочке преобразований.

In [ ]:
import pandas as pd
staff = pd.DataFrame({'hours':[5,8,3], 'rate':[1000,1200,800]})
with_salary = staff.assign(payment=lambda x: x.hours*x.rate)
print(with_salary)
print(staff.columns.tolist())  # исходник без payment

## 2. np.where и np.select
`numpy.where(condition, if_true, if_false)` — векторный аналог `if/else`, когда для каждой строки есть два варианта. Для нескольких непересекающихся условий используйте `np.select([condition1,...],[value1,...],default=...)`. **Порядок условий важен**: `np.select` выбирает первое истинное условие. Проверяйте, что итоговый тип удовлетворяет задаче, особенно при смешивании строк, чисел и пропусков.

In [ ]:
import numpy as np
tickets = pd.DataFrame({'days_left':[1,5,15]})
tickets['urgency'] = np.select([tickets.days_left<=2,tickets.days_left<=7],['critical','soon'],default='normal')
print(tickets)

## 3. where и mask — оставить или заменить
`Series.where(cond, other)` оставляет исходное значение там, где условие истинно; иначе заменяет на `other`. `Series.mask(cond, other)` делает **наоборот** — заменяет там, где условие истинно. Эти методы удобны, если нужно ограничить значения, обработать аномалии или добавить категорию. Для изменения нескольких колонок внутри существующей таблицы рекомендуется единый `.loc` — это соответствует Copy-on-Write в pandas 3.x.

In [ ]:
speeds = pd.Series([20,130,85,155], name='speed')
print(speeds.where(speeds<=120, 120))  # ограничить сверху
print(speeds.mask(speeds<30, 0))       # значения ниже 30 заменить нулём

## 4. map, replace, ошибки несоответствия индексов
`Series.map(mapping)` сопоставляет значения словарю; отсутствующий ключ обычно даёт пропуск. `replace()` полезен для замены известных значений, сохраняя остальные. Не забывайте: pandas выравнивает Series по индексам при присваивании. Если индексы не совпадают, вы можете получить пропуски вместо ожидаемых значений.

**Проверьте себя:** какая разница между `where` и `mask`? Почему порядок правил в `np.select` важен? Когда `map` может вернуть пропуски?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m2_16 · amount after discount

Добавьте `line_total` = `price * qty * (1 - discount/100)` и `big` = `line_total >= 200`, оставив исходный объект без изменений.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m2_16: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_16 = expected_for("m2_16", df=df)


In [ ]:
# m2_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 12 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `line_total`, `big`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  line_total    big
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...       254.0   True
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка       222.3   True
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно       368.6   True
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок       317.0   True
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                        136.8  False
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None         0.0  False
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m2_16", result, _expected_m2_16)

### m2_17 · priority band

Добавьте строковый `priority`: `urgent` при qty >= 4, `normal` при qty от 2 до 3, иначе `low`. Другие колонки сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m2_17: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_17 = expected_for("m2_17", df=df)


In [ ]:
# m2_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `priority`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment priority
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...   normal
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка      low
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно   urgent
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок      low
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                    normal
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None      low
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m2_17", result, _expected_m2_17)

### m2_18 · capped price

Создайте колонку `capped_price`, где `price` ограничена сверху значением `cap`, используя векторную обработку.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `cap` — входной параметр (значение видно в примере); в этом примере: `150`.


**Выполнение:** используйте готовые переменные `df`, `cap`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m2_18: используются переменные df, cap

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

cap = 150

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_18 = expected_for("m2_18", df=df, cap=cap)


In [ ]:
# m2_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, cap
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `capped_price`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  capped_price
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...         127.0
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка         150.0
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно          97.0
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок         150.0
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                           57.0
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None         147.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m2_18", result, _expected_m2_18)

### m2_19 · status labels

Добавьте `status_ru`: new→Новый, paid→Оплачен, cancelled→Отменён, остальные значения (если будут) сохраните без изменения.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m2_19: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_19 = expected_for("m2_19", df=df)


In [ ]:
# m2_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `status_ru`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment status_ru
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...   Оплачен
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка     Новый
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно   Оплачен
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок   Оплачен
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                    Отменён
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None     Новый
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m2_19", result, _expected_m2_19)

### m2_20 · discount quality

Добавьте колонку `net_price` (`price*(1-discount/100)`), но для статуса cancelled установите `net_price=0`. Возвращайте новый DataFrame.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m2_20: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_20 = expected_for("m2_20", df=df)


In [ ]:
# m2_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `net_price`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  net_price
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...     127.00
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка     222.30
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно      92.15
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок     317.00
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                        0.00
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None     147.00
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m2_20", result, _expected_m2_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()